# Crawling

## Overview

In this portion of the homework, you will crawl Google Jobs to collect job descriptions for later cleaning (HW-3.2 and HW-3.3) and EDA (HW-3.4).

We will use the [SerpApi](https://serpapi.com/) service to query Google Jobs. SerpApi is a paid API, but its free tier is more than enough for this homework. The API lets you search Google programmatically, which has a wealth of practical applications.

You will need an API key from SerpApi. It is free to sign up, and you should not enter any payment information.

https://serpapi.com/manage-api-key

At the time of writing, the free plan gives you **250 searches per month**, with a throughput cap of **50 searches per hour**. Each Google Jobs search returns up to 10 job descriptions.

This portion of the homework relies on limited API search resources, so prototype carefully. Once you are sure your code is working 100%, run it one last time for your final "production" crawl.

Budget for this assignment:

* About 10 searches for prototyping and debugging
* About 90 searches for the production run (30 job titles x 3 pages)
* The remaining searches are a safety buffer. Do not burn them on careless re-runs.

Make sure to save the outputs, and back them up (e.g., commit them to your GitHub repo) after the final run, in case you delete them by mistake.

We will use SerpApi's official Python client, which can be installed with

`pip install serpapi`

`Note`: older tutorials use the legacy `google-search-results` package (`from serpapi import GoogleSearch`). Both packages install a module named `serpapi`, so do not install both in the same environment. If you previously installed the legacy package, run `pip uninstall google-search-results` first.

The following are additional useful reference resources

* [https://serpapi.com/google-jobs-api](https://serpapi.com/google-jobs-api): parameters and response format for the Google Jobs engine
* [https://serpapi.com/integrations/python](https://serpapi.com/integrations/python): the Python client
* [https://serpapi.com/pricing](https://serpapi.com/pricing): current free-tier limits

An example of a saved search result is provided in `data/example-out-AI-Solutions-Architect-2024-10-20-15-41-05.json`. Open it to see the structure of the response before you spend any searches.

## Starter code

Here is some starter code:

`Note: uule parameter`

The `uule` parameter is an encoded location parameter used in Google search queries. It encodes the geographic location from which the search is being conducted, which influences the results to be more relevant to that location.

We set it to `'w+CAIQICINVW5pdGVkIFN0YXRlcw'`, which is an encoded string representing the United States. This simulates searches as if they are being conducted from that location.

SerpApi also accepts a human-readable `location` parameter (e.g., `'location': 'United States'`) instead. Use one or the other, not both.

In [1]:
import json
import os
import time

import serpapi

Save your API key in a centralized location outside of your project folder, e.g. `~/.api-keys.json`, and read it in with `json`. The file might look like:

```json
{
    "serpapi": "your-serpapi-key",
    "openai": "your-openai-key"
}
```

`IMPORTANT`: Never paste API keys directly into a notebook, never print them, and never commit them to GitHub. Keys that end up in a notebook output cell or a commit are effectively public.

In [6]:
with open(os.path.expanduser("~/Downloads/DSAN_5000/.api-keys.json")) as f:
    keys = json.load(f)
API_KEY = keys["serpapi"]

client = serpapi.Client(api_key=API_KEY)

Optional: SerpApi's account endpoint is free and does not count against your search quota, so you can use it to check how many searches you have left at any time. (Don't print the whole response, it includes your API key.)

In [7]:
account = client.account()
print("Searches left this month:", account.get("plan_searches_left"))
print("Total searches left:", account.get("total_searches_left"))

Searches left this month: 250
Total searches left: 250


Be careful, don't run the following cells too many times while debugging and prototyping, or you will use up your free searches.

In [8]:
search_query = "data science"
params = {
    "engine": "google_jobs",                  # SerpApi search engine
    "q": search_query,                        # search query
    "uule": "w+CAIQICINVW5pdGVkIFN0YXRlcw",   # encoded location (USA)
    "hl": "en",                               # language of the search
    "gl": "us",                               # country of the search
}

Let's do one search and explore the output.

In [9]:
try:
    result_dict = client.search(params).as_dict()   # search runs on the SerpApi backend; JSON -> Python dict
except serpapi.HTTPError as e:
    print("ERROR FOUND IN SEARCH:", e)
    result_dict = {}

if "error" in result_dict:
    print("ERROR FOUND IN SEARCH:", result_dict["error"])

In [10]:
for result in result_dict.get("jobs_results", []):
    print(result)

{'title': 'Data Scientist II, Analytics', 'company_name': 'Expedia Group', 'location': 'Seattle, WA', 'via': 'Expedia Group | Careers', 'job_highlights': [{'title': 'Qualifications', 'items': ['Experience applying statistical or analytical methods to business, product, customer, or partner questions', 'Experience with SQL and proficiency in Python, R, or a similar programming language', 'Understanding of descriptive statistics, inferential analysis, experiment design, and statistical concepts', 'Experience not just using AI but also validating its work and end products', 'Experience cleaning, joining, validating, and analyzing structured data from multiple sources', 'Ability to translate business questions into measurable metrics and analytical approaches', 'Ability to communicate insights clearly through written summaries, visualizations, dashboards, and presentations', 'Ability to manage small projects and collaborate with technical and non-technical stakeholders', 'Bachelor’s degree

In web crawling, pagination involves retrieving data across multiple pages of results. It helps manage large datasets by fetching a limited number of results per request, enabling efficient data extraction without overwhelming the server or exceeding resource limits.

You can use pagination to get more results for a given search, however you need to start where the last search left off.

You do this by adding the `next_page_token` to the `params` dictionary.

You get the latest `next_page_token` from `result_dict["serpapi_pagination"]`. If that key is missing, there are no more pages for that query.

`Note`: Google Jobs no longer supports the older `start` (offset) or `num` parameters. Pagination is done only with `next_page_token`, and each page returns at most 10 results.

If you don't paginate and search 10 times, you will just get the same first 10 results over and over again.

In [11]:
print(result_dict["serpapi_pagination"], "\n")
print(result_dict["serpapi_pagination"]["next_page_token"])

{'next_page_token': 'eyJmYyI6IkVzd0VDb3dFUVVwcFZEUjBTVmxZVVZvd1dEaG9NM1EyZHpSMWJGaEhZbkJqYUdSb2JXVkZSMEpRWW5sRlNrdFBjSGd5UTA5Rk0wcEViMWxGWmtwT1VXWnNkek5aWkdVek1HRlRWMk5pWlVKdlMweG5aRFV0UWtneFVtcFRjWEJ3VDBwdVR6VkNTVm96VWxwSlpqVkJObmhKV2toSlkxbDRTa2w2YkVGRFRqTTBTRGc1VGpKUlMyRmpia05uVm1sRE9HNXJlRU00Y0RKa1gyWkxUMHRPTFVNMVNHSklZbmxRTlRSb1dGRkJWMEZwZUdGQ2FXOVVhbXRzZVhSWE5FUjJYelpKU0dOYWNFdFlhM1V3UkcxellVdHFTMjgwUm5CTFFrMTNiMkV5T1hkRFJYVnVjMXBKYTBOSlh6WklPRTlVUTNGRGNERjFWVjh4VVV0TlN6SmhUazlHWkVoRE1WbEJka293TkdWbmNuSkpVVXBPYWpZeE1VUXhRWHBGZVVkVlQzaE1OMDVwYVZoZmNGOTBSWEl6TjBoVlQxSXhTblp2WTFWa2ExWnpZWFJoV21ob1FXWnhRVU5KYWtJeWVtZ3pObTlPY1VkRVUxaFhSRGxZV25VM09HSktSbDk1YlVsWVlXNTZUVlJVUkdoZk0ycEdXRWRKZVVwTmJXUmxlR3RxYzJ0MWNsVkZjbUp4Um5wYVgwSm5RVlZmYTFnMGFsZDRhbEJ2WVc5VlRYbEtZbkYyT1c1WmRUaHdWREJtYzNsSlJuVmFNRzFMTW5sWWNucHdZbTFLUTA1T1gwUmpSbEZEYlRWQmVFZFlUSHBUUVdvNWQxWjBhak5mTmpSR1pGQlJVRE14UTJSd1oxZEphbmxTZG1sUGEydzBaa1ZoYUdjdGJFUllUbEZLYmtaUVltaDVZV1E0TXpCYWQxRXhWalIxTjBNU0YxUnZVR0poY1V0clRVNU1TV3R

Let's do one more search, but this time with pagination, starting where the last search left off.

In [12]:
params = {
    "engine": "google_jobs",
    "q": search_query,
    "uule": "w+CAIQICINVW5pdGVkIFN0YXRlcw",
    "hl": "en",
    "gl": "us",
    "next_page_token": result_dict["serpapi_pagination"]["next_page_token"],
}

In [13]:
try:
    result_dict = client.search(params).as_dict()
except serpapi.HTTPError as e:
    print("ERROR FOUND IN SEARCH:", e)
    result_dict = {}

if "error" in result_dict:
    print("ERROR FOUND IN SEARCH:", result_dict["error"])

In [14]:
for result in result_dict.get("jobs_results", []):
    print(result["title"], ":", result["company_name"])

Data Scientist : Asurion
Data Science Analyst III -  Enterprise Data & Analytics - Digital and Technology Partners - Hybrid/Onsite : Mount Sinai Health Systems
Applied Data Scientist - AI Enablement (Transportation) : HDR
Data Science Program Lead II : AbbVie
NASA Human Spaceflight Artificial Intelligence (AI) Software Engineer and Data Scientist : CACI
Data Scientist : HCA Healthcare
Director of Data Science - Online Search and Recommendations : The Home Depot
Experienced Data Scientist for Innovative Technology Development and Strategic Decision Making – nexawork : Hireloom
Data Analytics Director : Pacific Life


# Utility function

Create a utility function that searches Google Jobs and saves the results to a file.

Here is one sketch of what the function might look like:

- Imports the current date and time using `datetime`.
- Defines `search_google_jobs` to perform a Google Jobs search with a default or custom query.
- Accepts parameters for the search query, pagination token, and verbosity.
- Sets search parameters like location, language, and search engine.
- Appends the pagination token if provided.
- Creates a timestamped output filename based on the query and time, e.g. `data/machine-learning-engineer-2026-10-20-15-04-53.json`.
- Makes sure the output directory exists (`os.makedirs("data", exist_ok=True)`).
- Does a search and data extraction, with error handling.
- Optionally prints a short summary if `verbose` is `True` (job titles and companies, NOT the full params dict, which contains your API key) and saves the results to a JSON file.
- Returns the `next_page_token` for pagination, or `None` if there are no more pages or an error occurred.

In [31]:
# INSERT CODE HERE
import datetime

def search_google_jobs(search_query="Data Scientist", next_page_token = None, verbose = True):
    params = {
    "engine": "google_jobs",
    "q": search_query,
    "uule": "w+CAIQICINVW5pdGVkIFN0YXRlcw",
    "hl": "en",
    "gl": "us"
    }

    if next_page_token != None:
        params["next_page_token"] = next_page_token

    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    output_name = f"{search_query.replace(' ', '-')}-{timestamp}.json"
    os.makedirs("data", exist_ok = True)
    output_path = os.path.join("data", output_name)

    try:
        result_dict = client.search(params).as_dict()
    except serpapi.HTTPError as e:
        print("ERROR FOUND IN SEARCH:", e)
        return {}
    
    if "error" in result_dict:
        print("ERROR FOUND IN SEARCH:", result_dict["error"])
        
    
    if verbose:
        for result in result_dict.get("jobs_results", []):
            print(result["title"], ":", result["company_name"])
            
        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(
                {"job_results": result_dict.get("jobs_results", [])},
                file,
                indent=4,
                ensure_ascii=False,
            )

#     if result_dict["serpapi_pagination"]["next_page_token"] is None:
#             print("ERROR FOUND IN SEARCH:", result_dict["error"])              


    return result_dict.get("serpapi_pagination", {}).get("next_page_token")

        



In [32]:
next_page_token = search_google_jobs(search_query="machine learning engineer", verbose=True)

Machine Learning Engineer - AI Trainer : AfterQuery Experts
Sr. Machine Learning Engineer, Medical Imaging : Intuitive Surgical
Staff Machine Learning Engineer - Mapping : General Motors
Computer Vision / Machine Learning Engineer : ExecutivePlacements.com
AI Engineer 5 (AI Foundations) : Capital One
Machine Learning Engineer : University of Illinois
Sr Machine Learning Engineer : Walt Disney Attractions Technology LLC
Machine Learning Infrastructure Engineer : Bright Vision Technologies
Senior Machine Learning Engineer - Content and Contributor Intelligence (Hybrid - United States) : Yelp
Lead Machine Learning Engineer Job Details | Capgemini : Capgemini


In [33]:
next_page_token

'eyJmYyI6IkV2WUVDcmNFUVVwcFZEUjBUREZVTUU5a1dVWlZaSEp2ZDJsM1prcHBabkJpYzBsM01qUklaVWd4Wm1NdFJHMWFTbkl6VDFwUVZsRk5MVkp1VUc5WWNFVkROM2xtT1U4eGRqSm1XbEJKZERCU1kwb3RhVXR4Y0Y4MVdtTTNhMEYyWDJWU1gzTTBOWEZEYVdjMVJWcHlOa0Z1YW5aak9FazRUREZFUW01TFdHTk1UMkUwV2xWcFozVXpjSEJaYkZWVlgzQkVhVmQyUlhoTlVFeFhkVWhqTVdvd1RsbElXR2RZTmtoNmFHNURlbU40VkdwWlJsZDRiRkJrWmpCRFMwdHZZbFp6WjBoV2VDMVlZV1ZxUW1wd1VqRktPWEZDTm5wRGVUTjFkRXhaVmxkdlJUVnhUbkZKVkhKb00ycDBWelF5YzNBMWJUSkxaRm8zWm01aFJEaFhWbGQ1Y0hwdWNWWkZlWGxHT0VkSk1WaE1kVzVJUVZwRFpsTlBNSHBhWWtGbFJWOUVRMDAyZFZCZmMwdE1iMWQ1ZEhGU1pFTmxiWFp3UTFGYWFXOWFZbms0WnpCVlRHRktTMUpmVkc5U1ZVNW9kVEYwUlhOelEydDFPWGhGZFY5eVVXRlhWRjlrVlRGSFowVkxaWEUyVldVdE9WQkdjR054YW10Sk5EZEJUamcxWWxGVFZqRnhhV0V6YkZCd1FsVmlUbE5VYjNGSlJIZFdOamR0VGpkcU9WWk1XV1JDU21KcFQxVkpaREJwVVRoVFNFcGlWbTFSVW5KVFUycHphV1o2VW5sRlJIbHVaR0pRU21GMVQwNXJNWFprVFdwUE9YZHdiV3B0VTNwMmVFRlpVRmhUVEdGaGFtMUxPWEJpTm5CVlFYRk1XbTFEVDFad2VsOWlTamhmVGxkUVFtaHZjalZSYkRkUFEzUlpNemhKT0daek5FRlllVlpCWlhsNWNEWk1SR3BuUjA5SVVEUjVPQzB6Wnp

In [34]:
search_google_jobs(search_query="machine learning engineer", next_page_token=next_page_token, verbose=True)

Machine Learning Engineer : THE DOW CHEMICAL COMPANY
Principal Machine Learning Engineer – Wafer Fabrication : Coherent Corp.
Senior Machine Learning Engineer, Maneuvering Tech : Waymo
Senior Machine Learning Engineer/Scientist- Identity & Trust : remitly
Principal Machine Learning Engineer, Foundation Models, AI for Drug Discovery : Genentech
Audio Machine Learning Intern/Co-op : Bose Corporation
NASA Human Spaceflight Artificial Intelligence (AI) Software Engineer and Data Scientist : CACI
Lead Machine Learning Engineer, Enterprise Platforms @ Capital One : 7Fold
Artificial Intelligence/Machine Learning (AI/ML) Engineer (2026-0238) : Acclaim Technical Services
Sr Machine Learning Engineer : Disney Experiences


'eyJmYyI6IkVvd0ZDc3dFUVVwcFZEUjBTbWhHVFdwcFNFcG5ZakV0VlRselYwbzVSa3A0T1VOS1dXcHpOMXBMV1RWSU1reHBhRUUzUjBrMFkyeHVkMnBGY1dSQ1VuZFRkemhSYlZkWFFucDZYMFIxUldOdVNFRmtVWEYzVVZSSVJTMXNOR1pGZEZaU1YwTkRlazExT1ZoMk1XZHdaemw2UjNoQmJGTktURlJpUkRKV1FtSnFkWEpCVkdOQ1dqWnhOak13VUZsYVpIRm9RMU10TnpnMlRDMXJjMjFSYmpsWmRVSmtURTFOYmpaU1ozWkxWV2RPWlMxdldrcFphMnAxY0UxVVVXTnpabFF0VG1GbWVua3pWR3gxY3pGaGRqVTVZalpzYm1OZlprUlpXVzkyVUdKME5reHhPREJNWnpaeVZGOVdTMGt4VDNGMGJYZEdTbFI0V1hOdFZEbEJZMUE1V25WU2VFTkhXRFpUV2w5RldFdzBjbTV0YzI5WVYzUmtSMWxLYkVSR1lqWnFhVVkxTVVoeFdXUjBiM1ZuVEdwMlEyY3haV2t0ZFVkc0xUVk1ZbkZOZUVaWVdtcE9URzlQTm1WMlNGbFVZakJsTmpGUlNucDVOMEozYVVKamFsbG5lRkJXVWxveWRsWktiRkJGZGpOVGR6aG1UVm8wVjI0dGVGQXdaR2xMZUhCU00zcHJNVGxaUWpCM2IwWTRjemc1ZUdKVlZqWm1NMHQ2WmxZeVQySk5ibGhVVXpWR1gwOXRlbFowZFVWWlZURjBVakZPUm1WM1QwUXlVbXBuV21OVWJ6aHRSVEl5YlVWSU5tbGFRM0ZtTW04eVNHazJaR0ZHVFZKbFVXVmZSMDR6WjFSR1QyRldWVzU2VkRKdmFuVkVZa3B1ZW5CRGNqZFJZMDF6UzFKRVlVNXNRV3h1TkZwZllXRmpObFp1V1ZaTFNIVm5PVTB5WWpSd09WUlJZVEJEZWpKcFZWZENVbWM0Vmt

Read back the most recent file your function saved, to confirm that it was written correctly.

In [35]:
import glob

latest_file = max(glob.glob("data/*.json"), key=os.path.getmtime)
print(latest_file)

with open(latest_file) as f:
    saved = json.load(f)

for result in saved.get("jobs_results", []):
    print(result["title"], ":", result["company_name"])

data/machine-learning-engineer-20261023-130032.json


`Checkpoint`: this is a good point to commit your work (`git add`, `git commit -m "..."`), if you haven't already.

## Iterate over job titles

The following 30 titles reflect a wide range of roles that leverage data science and machine learning skills across industries and specialties. You may swap a few of them for titles that interest you more, but keep the total at about 30 so that you stay within your search budget.

In [ ]:
job_titles = [
    "Data Scientist",
    "Machine Learning Engineer",
    "Artificial Intelligence Specialist",
    "Data Analyst",
    "Business Intelligence Analyst",
    "Research Scientist (AI-ML)",
    "Deep Learning Engineer",
    "NLP Engineer (Natural Language Processing)",
    "Computer Vision Engineer",
    "Data Engineer",
    "Applied Scientist",
    "Quantitative Analyst (Quant)",
    "AI Solutions Architect",
    "Statistician",
    "Big Data Engineer",
    "Data Science Consultant",
    "Automation Engineer",
    "Analytics Manager",
    "Operations Research Analyst",
    "Robotics Engineer",
    "Bioinformatics Data Scientist",
    "Financial Data Scientist",
    "Customer Insights Analyst",
    "Marketing Data Analyst",
    "Data Strategy Manager",
    "Cloud AI Engineer",
    "Computational Scientist",
    "Fraud Detection Specialist",
    "Risk Analyst",
    "Data Architect",
]

print(len(job_titles), "titles x 3 pages =", len(job_titles) * 3, "searches")

For each job title, do three searches using pagination. This will result in around 30 jobs per title (assuming at least 30 jobs exist for that title). Save each search result to its own file.

`Rate limits`: the free plan allows at most 50 searches per hour, and the full crawl is about 90 searches. Either

* pace your requests, e.g. `time.sleep(75)` between searches (the full crawl then takes about 2 hours, so let it run in the background), or
* split the job titles into batches of at most 15 titles (45 searches), and run one batch per hour.

Also make your loop robust: if a search fails or returns no `next_page_token`, move on to the next title instead of crashing, and skip titles whose results are already saved in `data/` so that a re-run does not repeat searches you already paid for.

Now insert code to iterate over the job titles and perform the searches.

Be very careful, this needs to be 100% correct before running it, otherwise you will burn through your free searches.

I recommend doing just one iteration of the loop as a trial run. If that looks good, do the next iteration and carefully check the results. If everything looks good, then run the remaining 28 iterations.

Note: sometimes pagination returns fewer than 10 results, so you may end up with slightly fewer than 30 results per title, e.g. 25 to 30.

Remember to clean the job titles before using them in filenames, i.e. remove or replace characters like spaces, `/`, or `()`.

When the crawl is finished, commit the `data/` folder so that your raw data is backed up.

In [ ]:
# INSERT CODE HERE